# ICT Daily-Bias Corpus — Transcript Extraction + Per-Video Concept Maps

**Scope (source-material ingestion only):** fetch transcripts for the two daily-bias teaching
videos in `bias_manifest.csv`, normalize them to a structured timestamped schema, and produce
**one concept map PER VIDEO** of raw timestamped excerpts. **No interpretation, no
summarization, no bias-rule synthesis, no reconciliation between the two creators.** Output is
raw wording with timestamps so a human (the ICT strategy chat) can build the bias spec
downstream.

This corpus is **daily-bias only** and stays **distinct** from the 2022 mentorship
midnight-open / Power 3 maps — it is never merged, cross-referenced, or folded in.

The reusable, network-free parsing/tagging/windowing logic lives in
`quant_research.utils.transcript_ingest` and is unit-tested in
`tests/utils/test_transcript_ingest.py`. This notebook is just orchestration.

## Dependencies (approved + installed during Phase A — do NOT reinstall)

- `youtube-transcript-api` — primary fetch (manual captions, then its generated track).
- `yt-dlp` (CLI) — fallback for auto-subs via `--write-auto-subs --skip-download`.

`whisper` is intentionally **not** used. If BOTH backends fail for a video, the run STOPs for
that video, logs it to `results/bias_concept_map/_missing.csv`, and prints an operator request
to drop a transcript at
`data/raw/ict_sources/bias_transcripts/<video_id>.(json|txt|vtt)` and re-run.

## Pipeline

1. **Fetch** — operator-supplied file (if present) first, else `youtube-transcript-api`, else
   `yt-dlp` fallback. Idempotent (skip videos whose normalized JSON already exists).
2. **Normalize** — one JSON per video at `data/processed/transcripts/bias/<video_id>.json`
   (LOCAL / gitignored — full third-party content).
3. **Concept map** — the bias lexicon scanned against each video separately, emitting one
   per-video map with top-level `creator`/`role`/`video_id`/`primary` metadata plus `±2`
   adjacent segments of context per hit. Maps are written to `docs/ict_bias_concept_maps/`
   (TRACKED) so the bias spec resolves from a clean clone.

In [1]:
"""Setup: locate the repo root, wire imports, capture the git SHA, and define output paths."""

import shutil
import subprocess
import sys
from importlib import util as importlib_util
from pathlib import Path

# Locate repo root by walking up to the directory that holds pyproject.toml.
_here = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_here, *_here.parents) if (p / "pyproject.toml").is_file()),
    _here,
)
_src = REPO_ROOT / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from quant_research.utils import transcript_ingest as ti  # noqa: E402

# Bias corpus paths. RAW transcripts stay LOCAL (gitignored); concept MAPS are tracked.
BIAS_MANIFEST_PATH = REPO_ROOT / "data" / "raw" / "ict_sources" / "bias_manifest.csv"
OPERATOR_DIR = REPO_ROOT / "data" / "raw" / "ict_sources" / "bias_transcripts"
TRANSCRIPT_DIR = REPO_ROOT / "data" / "processed" / "transcripts" / "bias"
MISSING_CSV = REPO_ROOT / "results" / "bias_concept_map" / "_missing.csv"
MAP_DIR = REPO_ROOT / "docs" / "ict_bias_concept_maps"

TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)


def _git_sha() -> str:
    """Return the short git SHA for artifact provenance, or 'unknown' if unavailable."""
    try:
        out = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
        )
        return out.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


GIT_SHA = _git_sha()
HAS_YT_TRANSCRIPT_API = importlib_util.find_spec("youtube_transcript_api") is not None
HAS_YT_DLP = shutil.which("yt-dlp") is not None

print(f"Repo root:          {REPO_ROOT}")
print(f"Bias manifest:      {BIAS_MANIFEST_PATH}  (exists={BIAS_MANIFEST_PATH.is_file()})")
print(f"Operator drop dir:  {OPERATOR_DIR}")
print(f"Transcript out dir: {TRANSCRIPT_DIR}  (LOCAL / gitignored)")
print(f"Concept map dir:    {MAP_DIR}  (TRACKED)")
print(f"git SHA:            {GIT_SHA}")
print(f"youtube-transcript-api installed: {HAS_YT_TRANSCRIPT_API}")
print(f"yt-dlp CLI available:             {HAS_YT_DLP}")

Repo root:          C:\Users\chris\Documents\Trading\quant-research-lab
Bias manifest:      C:\Users\chris\Documents\Trading\quant-research-lab\data\raw\ict_sources\bias_manifest.csv  (exists=True)
Operator drop dir:  C:\Users\chris\Documents\Trading\quant-research-lab\data\raw\ict_sources\bias_transcripts
Transcript out dir: C:\Users\chris\Documents\Trading\quant-research-lab\data\processed\transcripts\bias  (LOCAL / gitignored)
Concept map dir:    C:\Users\chris\Documents\Trading\quant-research-lab\docs\ict_bias_concept_maps  (TRACKED)
git SHA:            75c84e5
youtube-transcript-api installed: True
yt-dlp CLI available:             False


## Load the bias manifest

Reads `data/raw/ict_sources/bias_manifest.csv` (columns:
`video_id, title, url, creator, role, corpus`). If the file is missing this stops with an
error — we do **not** guess video IDs. The second video id begins with a hyphen; it is always
handled as a manifest VALUE (never a bare CLI argument).

In [2]:
bias_rows = ti.read_bias_manifest(BIAS_MANIFEST_PATH)  # raises FileNotFoundError if absent
print(f"Loaded {len(bias_rows)} bias-manifest rows.\n")
for row in bias_rows:
    print(f"  {row.video_id:<12}  {row.creator:<8} role={row.role:<10} corpus={row.corpus}  {row.title}")

Loaded 2 bias-manifest rows.

  x4FJVEZUXhc   jadecap  role=primary    corpus=bias  The Only ICT Daily Bias Video You'll Ever Need
  -KKuZb5Z5aU   ttrades  role=supplement corpus=bias  Easy Daily Bias - Mechanical Framework


## Steps 1 + 2 — Fetch and normalize (operator-supplied fallback)

For each manifest row: skip if its normalized JSON already exists (idempotent). Otherwise, if an
operator-supplied transcript exists at
`data/raw/ict_sources/bias_transcripts/<video_id>.(json|txt|vtt)`, parse it into the segment
schema (`source="operator_supplied"`). Otherwise fetch via `youtube-transcript-api`, then
`yt-dlp`. If BOTH network backends fail, the video is logged to
`results/bias_concept_map/_missing.csv` and a clear operator request is printed — no audio
transcription is attempted (whisper is not approved).

In [3]:
import csv
import json
import time

REQUEST_DELAY_S = 1.5  # polite pause between network fetches to avoid YouTube rate-limits


def _find_operator_file(video_id: str) -> Path | None:
    """Return an operator-supplied transcript file for ``video_id`` if one exists."""
    for ext in ("json", "vtt", "txt"):
        candidate = OPERATOR_DIR / f"{video_id}.{ext}"
        if candidate.is_file():
            return candidate
    return None


def _parse_operator_file(path: Path) -> list[dict]:
    """Parse an operator-supplied .json/.vtt/.txt file into raw segment dicts.

    .json may be either an already-normalized transcript ({"segments": [...]}) or a bare list
    of snippet dicts. .vtt is parsed with the module's tested ``parse_vtt``. .txt has no
    timestamps, so each non-empty line becomes a segment with a synthetic 1s-spaced start.
    """
    suffix = path.suffix.lower()
    if suffix == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, dict) and "segments" in data:
            return list(data["segments"])
        return list(data)
    if suffix == ".vtt":
        return ti.parse_vtt(path.read_text(encoding="utf-8", errors="replace"))
    # .txt: no timestamps available; index lines as synthetic seconds for stable ordering.
    lines = [ln.strip() for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    return [{"text": ln, "start": float(i), "end": float(i)} for i, ln in enumerate(lines)]


def _log_bias_missing(row, reason: str) -> None:
    """Append a row to the bias _missing.csv log, writing the header if the file is new."""
    MISSING_CSV.parent.mkdir(parents=True, exist_ok=True)
    write_header = not MISSING_CSV.exists()
    with MISSING_CSV.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        if write_header:
            writer.writerow(ti.BIAS_MISSING_CSV_COLUMNS)
        writer.writerow([row.video_id, row.title, row.url, row.creator, reason])


fetched_count = skipped_count = missing_count = 0

for row in bias_rows:
    out_path = TRANSCRIPT_DIR / f"{row.video_id}.json"
    if out_path.exists():
        skipped_count += 1
        print(f"  [skip]      {row.video_id}: normalized JSON already exists")
        continue

    operator_file = _find_operator_file(row.video_id)
    if operator_file is not None:
        source, raw_segments = "operator_supplied", _parse_operator_file(operator_file)
    else:
        time.sleep(REQUEST_DELAY_S)
        try:
            result = ti.fetch_transcript(row.video_id)  # video_id is a VALUE, never a CLI flag
        except Exception as exc:  # never abort the whole run on one failure
            result = None
            print(f"  [fetch-error] {row.video_id}: {exc!r}")
        if result is None:
            _log_bias_missing(row, "no_transcript")
            missing_count += 1
            print(
                f"  [MISSING]   {row.video_id}: both backends failed.\n"
                f"    Transcript retrieval failed for {row.creator} ({row.video_id}). Please "
                f"supply the transcript at\n"
                f"    data/raw/ict_sources/bias_transcripts/{row.video_id}.(json|txt|vtt) and "
                f"re-run."
            )
            continue
        source, raw_segments = result

    document = ti.normalize_bias_transcript(
        video_id=row.video_id,
        title=row.title,
        url=row.url,
        creator=row.creator,
        role=row.role,
        corpus=row.corpus,
        source=source,
        raw_segments=raw_segments,
    )
    ti.write_json(out_path, document)
    fetched_count += 1
    print(f"  [{source:<16}] {row.video_id}: {len(document['segments'])} segments")

print(
    f"\nFetch summary: {fetched_count} written, {skipped_count} skipped (already present), "
    f"{missing_count} missing."
)
if missing_count:
    print(f"Missing videos logged to: {MISSING_CSV}")
    print("Supply the file(s) above and re-run this notebook to complete the corpus.")

  [auto_subs       ] x4FJVEZUXhc: 383 segments


  [auto_subs       ] -KKuZb5Z5aU: 490 segments

Fetch summary: 2 written, 0 skipped (already present), 0 missing.


## Step 3 — Per-video concept maps (kept DISTINCT)

Load each normalized bias transcript and scan the **bias lexicon** against it **separately**.
The two maps are never merged and conflicts are never reconciled — that is the downstream ICT
chat's job. Each map carries top-level `creator`/`role`/`video_id`/`primary` metadata (JadeCap =
`true`, TTrades = `false`) plus the git SHA, and each hit includes `±2` adjacent segments of
context. Pure keyword windowing — no paraphrasing, no LLM summarization.

Output (TRACKED): `docs/ict_bias_concept_maps/jadecap_primary.json` and
`docs/ict_bias_concept_maps/ttrades_supplement.json`.

In [4]:
MAP_FILENAMES = {
    ("jadecap", "primary"): "jadecap_primary.json",
    ("ttrades", "supplement"): "ttrades_supplement.json",
}

written_maps = []
for row in bias_rows:
    transcript_path = TRANSCRIPT_DIR / f"{row.video_id}.json"
    if not transcript_path.exists():
        print(f"  [skip-map]  {row.video_id}: no normalized transcript yet (see _missing.csv)")
        continue
    with transcript_path.open("r", encoding="utf-8") as handle:
        transcript = json.load(handle)

    bias_map = ti.build_bias_concept_map(transcript, git_sha=GIT_SHA)  # default BIAS_LEXICON, ±2
    filename = MAP_FILENAMES.get((row.creator, row.role), f"{row.creator}_{row.role}.json")
    out_path = ti.write_json(MAP_DIR / filename, bias_map)
    written_maps.append(out_path)

    total_hits = sum(len(hits) for hits in bias_map["concepts"].values())
    print(f"  {row.creator:<8} role={row.role:<10} primary={bias_map['primary']!s:<5} -> {filename}")
    for concept, hits in bias_map["concepts"].items():
        print(f"      {concept:<20} {len(hits):>4} hits")
    print(f"      total: {total_hits} hits across {len(bias_map['concepts'])} concept groups\n")

print("Concept maps written (TRACKED):")
for path in written_maps:
    print(f"  {path}")

  jadecap  role=primary    primary=True  -> jadecap_primary.json
      bias_core              33 hits
      htf_reference          18 hits
      liquidity_targets       9 hits
      pd_array               21 hits
      midpoint_validation     2 hits
      structure               0 hits
      timeframe_nesting       6 hits
      no_clear_bias           2 hits
      total: 91 hits across 8 concept groups

  ttrades  role=supplement primary=False -> ttrades_supplement.json
      bias_core               9 hits
      htf_reference          15 hits
      liquidity_targets      16 hits
      pd_array               12 hits
      midpoint_validation     0 hits
      structure               0 hits
      timeframe_nesting       1 hits
      no_clear_bias          16 hits
      total: 69 hits across 8 concept groups

Concept maps written (TRACKED):
  C:\Users\chris\Documents\Trading\quant-research-lab\docs\ict_bias_concept_maps\jadecap_primary.json
  C:\Users\chris\Documents\Trading\quant-research